# 04 PPO 实战：CartPole-v1

**目标**：从零实现 PPO-Clip。这一份代码基本可以直接迁移到任何离散动作环境。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                    nn.Linear(hidden, hidden), nn.Tanh())
        self.actor = nn.Linear(hidden, n_actions)
        self.critic = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h).squeeze(-1)
    def act(self, x):
        logits, v = self.forward(x)
        dist = Categorical(logits=logits)
        a = dist.sample()
        return a, dist.log_prob(a), v

## GAE 计算

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$
$$\hat{A}_t = \sum_{l=0}^{T-t} (\gamma\lambda)^l \delta_{t+l}$$

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages = np.zeros(len(rewards), dtype=np.float32)
    gae = 0
    for t in reversed(range(len(rewards))):
        next_value = values[t+1] if t+1 < len(values) else 0
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages[t] = gae
    returns = advantages + np.array(values[:-1] if len(values) > len(rewards) else values)
    return advantages, returns

In [ ]:
ENV_ID = 'CartPole-v1'
GAMMA = 0.99
LAM = 0.95
CLIP = 0.2
LR = 3e-4
EPOCHS = 10
BATCH = 64
ROLLOUT = 2048
TOTAL_STEPS = 50_000
ENT_COEF = 0.01
VAL_COEF = 0.5

env = gym.make(ENV_ID)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

model = ActorCritic(obs_dim, n_actions).to(device)
opt = optim.Adam(model.parameters(), lr=LR)

obs, _ = env.reset(seed=0)
global_step = 0
ep_returns, ep_return_now = [], 0

while global_step < TOTAL_STEPS:
    obs_buf, act_buf, logp_buf, val_buf, rew_buf, done_buf = [], [], [], [], [], []
    for _ in range(ROLLOUT):
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            a, logp, v = model.act(obs_t)
        a_i = a.item()
        obs2, r, term, trunc, _ = env.step(a_i)
        done = term or trunc
        obs_buf.append(obs); act_buf.append(a_i); logp_buf.append(logp.item())
        val_buf.append(v.item()); rew_buf.append(r); done_buf.append(float(term))
        obs = obs2
        ep_return_now += r
        global_step += 1
        if done:
            ep_returns.append(ep_return_now); ep_return_now = 0
            obs, _ = env.reset()
    
    with torch.no_grad():
        last_v = model(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))[1].item()
    val_buf_full = val_buf + [last_v]
    advantages, returns = compute_gae(rew_buf, val_buf_full, done_buf, GAMMA, LAM)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    obs_t = torch.tensor(np.array(obs_buf), dtype=torch.float32, device=device)
    act_t = torch.tensor(act_buf, dtype=torch.long, device=device)
    logp_old = torch.tensor(logp_buf, dtype=torch.float32, device=device)
    adv_t = torch.tensor(advantages, device=device)
    ret_t = torch.tensor(returns, dtype=torch.float32, device=device)

    n = len(obs_buf)
    idx = np.arange(n)
    for ep_i in range(EPOCHS):
        np.random.shuffle(idx)
        for start in range(0, n, BATCH):
            mb = idx[start:start+BATCH]
            logits, v = model(obs_t[mb])
            dist = Categorical(logits=logits)
            logp_new = dist.log_prob(act_t[mb])
            entropy = dist.entropy().mean()
            ratio = (logp_new - logp_old[mb]).exp()
            surr1 = ratio * adv_t[mb]
            surr2 = torch.clamp(ratio, 1-CLIP, 1+CLIP) * adv_t[mb]
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss = ((v - ret_t[mb]) ** 2).mean()
            loss = policy_loss + VAL_COEF * value_loss - ENT_COEF * entropy
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step()
    
    if len(ep_returns) > 10:
        recent = np.mean(ep_returns[-10:])
        print(f'step={global_step:6d}  recent10_return={recent:.1f}  policy_loss={policy_loss.item():.3f}  value_loss={value_loss.item():.3f}')

env.close()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ep_returns, alpha=0.3)
if len(ep_returns) > 10:
    smooth = np.convolve(ep_returns, np.ones(10)/10, mode='valid')
    plt.plot(np.arange(9, len(ep_returns)), smooth, linewidth=2, label='smoothed')
plt.axhline(195, color='r', linestyle='--')
plt.xlabel('Episode'); plt.ylabel('Return'); plt.title('PPO on CartPole-v1')
plt.grid(True); plt.legend(); plt.show()

## 练习

1. 把环境换成 `LunarLander-v2`，调超参跑通
2. 改成连续动作（用 `gym.make('Pendulum-v1')`），策略输出 mean+std，用 `Normal` 分布
3. 加入 W&B 记录 KL(π_old || π_new)，看是否 < 0.02
4. 跑 5 个 seed，对比方差